In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn import metrics
from sklearn.linear_model import  LogisticRegression
import numpy as np
import random
import math
import time
import lightgbm as lgb


In [3]:
data = pd.read_csv("Bcard.txt")
data.head()

,obs_mth,bad_ind,uid,td_score,jxl_score,mj_score,rh_score,zzc_score,zcx_score,person_info,finance_info,credit_info,act_info
0,2018-10-31,0.0,A10000005,0.675349,0.144072,0.186899,0.483640,0.928328,0.369644,-0.322581,0.023810,0.00,0.217949
1,2018-07-31,0.0,A1000002,0.825269,0.398688,0.139396,0.843725,0.605194,0.406122,-0.128677,0.023810,0.00,0.423077
2,2018-09-30,0.0,A1000011,0.315406,0.629745,0.535854,0.197392,0.614416,0.320731,0.062660,0.023810,0.10,0.448718
3,2018-07-31,0.0,A10000481,0.002386,0.609360,0.366081,0.342243,0.870006,0.288692,0.078853,0.071429,0.05,0.179487
4,2018-07-31,0.0,A1000069,0.406310,0.405352,0.783015,0.563953,0.715454,0.512554,-0.261014,0.023810,0.00,0.423077


In [5]:
df_train = data[data["obs_mth"]!="2018-11-30"]
df_train.head()

,obs_mth,bad_ind,uid,td_score,jxl_score,mj_score,rh_score,zzc_score,zcx_score,person_info,finance_info,credit_info,act_info
0,2018-10-31,0.0,A10000005,0.675349,0.144072,0.186899,0.483640,0.928328,0.369644,-0.322581,0.023810,0.00,0.217949
1,2018-07-31,0.0,A1000002,0.825269,0.398688,0.139396,0.843725,0.605194,0.406122,-0.128677,0.023810,0.00,0.423077
2,2018-09-30,0.0,A1000011,0.315406,0.629745,0.535854,0.197392,0.614416,0.320731,0.062660,0.023810,0.10,0.448718
3,2018-07-31,0.0,A10000481,0.002386,0.609360,0.366081,0.342243,0.870006,0.288692,0.078853,0.071429,0.05,0.179487
4,2018-07-31,0.0,A1000069,0.406310,0.405352,0.783015,0.563953,0.715454,0.512554,-0.261014,0.023810,0.00,0.423077


In [6]:
val = data[data["obs_mth"]=="2018-11-30"]
val.head()


,obs_mth,bad_ind,uid,td_score,jxl_score,mj_score,rh_score,zzc_score,zcx_score,person_info,finance_info,credit_info,act_info
79831,2018-11-30,0.0,A10002345,0.123276,0.872117,0.723560,0.759074,0.184735,0.080376,-0.053718,0.047619,1.00,0.230769
79832,2018-11-30,0.0,A10003755,0.462460,0.157643,0.762271,0.481466,0.967006,0.780087,0.013863,0.023810,0.00,0.230769
79833,2018-11-30,0.0,A1000756,0.812642,0.400040,0.280942,0.099454,0.942880,0.588936,0.078853,0.023810,0.02,0.474359
79834,2018-11-30,0.0,A100085,0.007039,0.396036,0.857868,0.882255,0.345511,0.419969,-0.053718,0.047619,0.02,0.666667
79835,2018-11-30,0.0,A10008856,0.078063,0.291289,0.654864,0.528708,0.754482,0.732534,0.013863,0.023810,0.00,0.230769


In [7]:
df_train = df_train.sort_values(by="obs_mth", ascending=False)
df_train["rank"] = [i for i in range(df_train.shape[0])]


In [9]:
df_train.head()

,obs_mth,bad_ind,uid,td_score,jxl_score,mj_score,rh_score,zzc_score,zcx_score,person_info,finance_info,credit_info,act_info,rank
0,2018-10-31,0.0,A10000005,0.675349,0.144072,0.186899,0.483640,0.928328,0.369644,-0.322581,0.023810,0.00,0.217949,0
33407,2018-10-31,0.0,A2810176,0.146055,0.079922,0.250568,0.045240,0.766906,0.413713,0.013863,0.023810,0.00,0.269231,1
33383,2018-10-31,0.0,A2807687,0.551366,0.300781,0.225007,0.045447,0.735733,0.684182,-0.261014,0.071429,0.03,0.269231,2
33379,2018-10-31,0.0,A2807232,0.708547,0.769513,0.928457,0.739716,0.947453,0.361551,-0.128677,0.047619,0.00,0.269231,3
33376,2018-10-31,0.0,A2806932,0.482248,0.116658,0.286273,0.056618,0.047024,0.890433,0.078853,0.047619,0.00,0.269231,4


In [11]:
df_train["rank"] = pd.cut(df_train["rank"], bins=5, labels=[i for i in range(5)])
df_train.head()

,obs_mth,bad_ind,uid,td_score,jxl_score,mj_score,rh_score,zzc_score,zcx_score,person_info,finance_info,credit_info,act_info,rank
0,2018-10-31,0.0,A10000005,0.675349,0.144072,0.186899,0.483640,0.928328,0.369644,-0.322581,0.023810,0.00,0.217949,0
33407,2018-10-31,0.0,A2810176,0.146055,0.079922,0.250568,0.045240,0.766906,0.413713,0.013863,0.023810,0.00,0.269231,0
33383,2018-10-31,0.0,A2807687,0.551366,0.300781,0.225007,0.045447,0.735733,0.684182,-0.261014,0.071429,0.03,0.269231,0
33379,2018-10-31,0.0,A2807232,0.708547,0.769513,0.928457,0.739716,0.947453,0.361551,-0.128677,0.047619,0.00,0.269231,0
33376,2018-10-31,0.0,A2806932,0.482248,0.116658,0.286273,0.056618,0.047024,0.890433,0.078853,0.047619,0.00,0.269231,0


In [12]:
df_train["rank"].value_counts()

rank
0    15967
1    15966
2    15966
3    15966
4    15966
Name: count, dtype: int64

In [15]:
def lgb_test(train_x, train_y, test_x, test_y):
    clf =lgb.LGBMClassifier(boosting_type = 'gbdt',objective = 'binary',metric = 'auc',
    learning_rate = 0.2,n_estimators = 200,max_depth = 3,num_leaves = 20,
    max_bin = 45,min_data_in_leaf = 6,bagging_fraction = 0.6,bagging_freq = 0,
    feature_fraction = 0.8,
    )

    clf.fit(train_x, train_y, eval_set=[(train_x, train_y), (test_x, test_y)], eval_metric='auc')
    return clf, clf.best_score_["valid_1"]["auc"]

In [16]:
feature_list = ['td_score', 'jxl_score', 'mj_score','rh_score', 'zzc_score', 'zcx_score', 'person_info', 'finance_info','credit_info', 'act_info']
feature_importance_lst = []
ks_train_lst = []
ks_test_lst = []
auc_list = []

In [17]:
for rk in range(5):
    ttest = df_train[df_train["rank"] == rk]
    ttrain = df_train[df_train["rank"] != rk]

    train_x = ttrain[feature_list]
    train_y = ttrain["bad_ind"]
    test_x = ttest[feature_list]
    test_y = ttest["bad_ind"]

    model, auc = lgb_test(train_x, train_y, test_x, test_y)
    feature_importance_df = pd.DataFrame({"name": model.booster_.feature_name(),
                                          "importance": model.feature_importances_}).set_index("name")
    feature_importance_lst.append(feature_importance_df)
    auc_list.append(auc)

    y_pred_train = model.predict_proba(train_x)[:, 1]
    y_pred_test = model.predict_proba(test_x)[:, 1]

    fpr_train,tpr_train,threshold_train = roc_curve(train_y,y_pred_train)
    fpr_test,tpr_test,threshold_test = roc_curve(test_y,y_pred_test)

    train_ks = abs(fpr_train-tpr_train).max()
    test_ks = abs(fpr_test-tpr_test).max()

    ks_train_lst.append(train_ks)
    ks_test_lst.append(test_ks)

[LightGBM] [Warning] min_data_in_leaf is set=6, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Warning] min_data_in_leaf is set=6, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Info] Number of positive: 1054, number of negative

In [18]:
pd.concat(feature_importance_lst, axis=1).mean(1).sort_values(ascending=False)

C:\Users\mingw\AppData\Local\Temp\ipykernel_6264\2121322890.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  pd.concat(feature_importance_lst, axis=1).mean(1).sort_values(ascending=False)


name
mj_score        156.6
act_info        155.8
jxl_score       145.4
zcx_score       145.2
credit_info     144.8
rh_score        144.2
zzc_score       140.6
finance_info    135.4
td_score        133.0
person_info      71.8
dtype: float64

In [19]:
auc_list

[np.float64(0.7546322460094083),
 np.float64(0.7778534325804788),
 np.float64(0.8082090258129289),
 np.float64(0.7883993523058535),
 np.float64(0.7691780385154955)]

In [20]:
ks_train_lst

[np.float64(0.6196855550926607),
 np.float64(0.6133916865287659),
 np.float64(0.5839305975025277),
 np.float64(0.5997490388664464),
 np.float64(0.6128277550700191)]

In [21]:
lst = ['person_info','finance_info','credit_info','act_info']
train = data[data.obs_mth != '2018-11-30'].reset_index().copy()
evl = data[data.obs_mth == '2018-11-30'].reset_index().copy()
x = train[lst]
y = train['bad_ind']
evl_x = evl[lst]
evl_y = evl['bad_ind']
model,auc = lgb_test(x,y,evl_x,evl_y)
y_pred = model.predict_proba(x)[:,1]
fpr_lgb_train,tpr_lgb_train,_ = roc_curve(y,y_pred)
train_ks = abs(fpr_lgb_train - tpr_lgb_train).max()
print('train_ks : ',train_ks)
y_pred = model.predict_proba(evl_x)[:,1]
fpr_lgb,tpr_lgb,_ = roc_curve(evl_y,y_pred)
evl_ks = abs(fpr_lgb - tpr_lgb).max()
print('evl_ks : ',evl_ks)

[LightGBM] [Warning] min_data_in_leaf is set=6, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Warning] min_data_in_leaf is set=6, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Info] Number of positive: 1470, number of negative

In [22]:
bins = 20
temp_df = pd.DataFrame() # 准备空白的df
# 用训练好的模型, 输出测试集的违约率
temp_df['bad_rate_predict'] = model.predict_proba(evl_x)[:,1] # 模型预测的违约率
temp_df['real_bad']= evl_y.values # 真实的标签
temp_df = temp_df.sort_values('bad_rate_predict',ascending=False)
temp_df['num'] = [i for i in range(temp_df.shape[0])]
temp_df['num'] = pd.cut(temp_df['num'],bins = bins,labels=[i for i in range(bins)])

[LightGBM] [Warning] min_data_in_leaf is set=6, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0


In [23]:
report = pd.DataFrame()
# 每一组有多少1 bad标签 , 每一组有多少0 good标签
report['BAD'] =temp_df.groupby('num')['real_bad'].sum().astype(int)
report['GOOD'] =temp_df.groupby('num')['real_bad'].count().astype(int)-report['BAD']
# 累计求和 累计到这一组, 有多少1, 多少0
report['BAD_CNT'] = report['BAD'].cumsum()
report['GOOD_CNT'] = report['GOOD'].cumsum()
# 计算累计到当前组, 出现的1标签的比例
good_total = report['GOOD_CNT'].max()
bad_total = report['BAD_CNT'].max()
report['BAD_PCTG'] = round(report['BAD_CNT']/bad_total,3)
# 当前组 1标签比例
report['BAD_RATE'] = report.apply(lambda x:round(x['BAD']/(x['BAD']+x['GOOD']),3),axis = 1)
# 当前组ks
def cal_ks(x):
	# tpr = tp/tp+fn(所有的1标签)  fpr = fp/fp+tn (所有的0) GOOD_CNT
	ks = (x['BAD_CNT']/bad_total)-(x['GOOD_CNT']/good_total)
	return round(math.fabs(ks),3)
report['KS'] = report.apply(cal_ks,axis = 1)
print(report)

     BAD  GOOD  BAD_CNT  GOOD_CNT  BAD_PCTG  BAD_RATE     KS
num                                                         
0     90   709       90       709     0.274     0.113  0.229
1     30   769      120      1478     0.366     0.038  0.271
2     32   767      152      2245     0.463     0.040  0.320
3     40   758      192      3003     0.585     0.050  0.393
4     25   774      217      3777     0.662     0.031  0.420
5     16   783      233      4560     0.710     0.020  0.419
6     16   782      249      5342     0.759     0.020  0.418
7     12   787      261      6129     0.796     0.015  0.404
8      9   790      270      6919     0.823     0.011  0.381
9     15   784      285      7703     0.869     0.019  0.377
10     8   790      293      8493     0.893     0.010  0.351
11     7   792      300      9285     0.915     0.009  0.321
12     8   791      308     10076     0.939     0.010  0.295
13     2   796      310     10872     0.945     0.003  0.250
14     4   795      314 

In [24]:
def score(bad_prob):
    return 500+25*(math.log2((1-bad_prob)/bad_prob))
evl['bad_prob'] = model.predict_proba(evl_x)[:,1]
evl['score'] = evl['bad_prob'].apply(score)
evl['score'].describe()

[LightGBM] [Warning] min_data_in_leaf is set=6, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0


count    15975.000000
mean       656.043219
std         49.102979
min        355.572675
25%        622.157693
50%        650.216555
75%        686.736769
max        892.805773
Name: score, dtype: float64

In [25]:
from pyecharts.charts import *
from pyecharts import options as opts
from pylab import *
mpl.rcParams['font.sans-serif'] = ['SimHei']
np.set_printoptions(suppress=True)
pd.set_option('display.unicode.ambiguous_as_wide', True)
pd.set_option('display.unicode.east_asian_width', True)
line = (Line()
	.add_xaxis(report.index.values.tolist())
	.add_yaxis("分组坏人占比",list(report.BAD_RATE),yaxis_index=0,color="red",
	)
	.set_global_opts(title_opts=opts.TitleOpts(title="评分卡模型表现"),	)
	.extend_axis(
		yaxis=opts.AxisOpts(name="KS值",type_="value",min_=0,max_=0.5,position="right",
		axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="red")),
		axislabel_opts=opts.LabelOpts(formatter="{value}"),
		))
	.add_yaxis("KS",list(report['KS']),yaxis_index=1,color="blue",label_opts=opts.LabelOpts(is_show=False),
	)
)
line.render()

'F:\\6个月\\the road to machine learning\\1-financial risk management project\\data\\render.html'